# 🌍 FineTome → Kiswahili Translation Pipeline
### OpenAI Batch API · Quality Filtering · HuggingFace Hub Push

**Goal:** Translate a curated 20K subset of `mlabonne/FineTome-100k` to high-quality Swahili  
**Model:** GPT-4o-mini via OpenAI Batch API (50% cheaper, no rate limit pressure)  
**Output:** `lengai-lab/FineTome-20k-sw` — Swahili instruction dataset  
**Cost:** ~$0.15 | **Time:** 2–6 hours (submit and sleep)

---
```
1. Install & configure
2. Load & filter FineTome (best 20K rows)
3. Format as OpenAI Batch JSONL
4. Submit batch job → save batch ID
5. Poll until complete
6. Download & parse results
7. Quality validation
8. Push to HuggingFace Hub
```

## 1. Install & Configure

In [ ]:
%%capture
!pip install openai datasets huggingface_hub tqdm pandas

In [ ]:
import os

# ── KEYS (set as Colab secrets or env vars) ──────────────────────────────────
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-...")
HF_TOKEN       = os.environ.get("HF_TOKEN",       "hf_...")

# ── CONFIG ───────────────────────────────────────────────────────────────────
SOURCE_DATASET = "mlabonne/FineTome-100k"
TARGET_ROWS    = 20_000
HF_REPO        = "lengai-lab/FineTome-20k-sw"   # ← change to your HF username
MODEL          = "gpt-4o-mini"
MAX_TOKENS     = 1024
TEMPERATURE    = 0.3
BATCH_JSONL    = "batch_input.jsonl"
RESULTS_JSONL  = "batch_results.jsonl"

print(f"✅ Config ready | {TARGET_ROWS:,} rows | Model: {MODEL}")

## 2. Load & Filter FineTome

Select the best rows — remove code-heavy, too short, too long.

In [ ]:
from datasets import load_dataset
import re, random, json

print("Loading FineTome-100k...")
ds = load_dataset(SOURCE_DATASET, split="train")
print(f"Loaded {len(ds):,} rows | Columns: {ds.column_names}")
print("\nSample:", ds[0])

In [ ]:
def extract_turns(row):
    """Extract first human/gpt turn pair from ShareGPT format."""
    instruction, output = "", ""
    for turn in row.get("conversations", []):
        if turn["from"] in ("human", "user") and not instruction:
            instruction = turn["value"].strip()
        elif turn["from"] in ("gpt", "assistant") and not output:
            output = turn["value"].strip()
    return instruction, output

def is_code_heavy(text, threshold=0.3):
    patterns = [r"```[\s\S]*?```", r"def \w+\s*\(", r"import \w+",
                r"class \w+[:(]", r"<[a-zA-Z][^>]*>[^<]*</[a-zA-Z]+>"]
    code_chars = sum(len(m.group()) for p in patterns for m in re.finditer(p, text))
    return (code_chars / max(len(text), 1)) > threshold

def is_translatable(instruction, output):
    if not instruction or not output:         return False
    if len(output.split()) < 20:              return False  # too short
    if len(output.split()) > 600:             return False  # too long
    if is_code_heavy(instruction + output):   return False  # mostly code
    return True

print("Filtering...")
filtered = []
for i, row in enumerate(ds):
    instruction, output = extract_turns(row)
    if is_translatable(instruction, output):
        filtered.append({
            "id":          str(i),
            "instruction": instruction,
            "output":      output,
        })

print(f"After filtering: {len(filtered):,} rows (removed {len(ds)-len(filtered):,})")

# Sample evenly across dataset for topic diversity
random.seed(42)
if len(filtered) > TARGET_ROWS:
    step     = len(filtered) // TARGET_ROWS
    selected = [filtered[i] for i in range(0, len(filtered), step)][:TARGET_ROWS]
else:
    selected = filtered

print(f"✅ Selected {len(selected):,} rows for translation")

## 3. Format as Batch JSONL

In [ ]:
SYSTEM_PROMPT = """Wewe ni mtafsiri mtaalamu wa Kiingereza hadi Kiswahili sanifu.
Kazi yako ni kutafsiri maagizo na majibu ya AI kwa Kiswahili fasaha na asilia.

Kanuni:
1. Tumia Kiswahili sanifu na fasaha — si tafsiri ya neno kwa neno
2. Hifadhi maana kamili ya asili bila kupoteza taarifa yoyote
3. Maneno ya kitaalamu (AI, model, data, algorithm, n.k.) yanaweza kubaki Kiingereza
4. Nambari, fomula za hisabati, na msimbo wa kompyuta vibaki kama vilivyo
5. Jibu kwa JSON tu — bila maelezo, bila utangulizi, bila alama za ```

Muundo wa jibu:
{"instruction": "<tafsiri ya maagizo>", "output": "<tafsiri ya jibu>"}"""

def make_request(row):
    return {
        "custom_id": f"row-{row['id']}",
        "method":    "POST",
        "url":       "/v1/chat/completions",
        "body": {
            "model":           MODEL,
            "temperature":     TEMPERATURE,
            "max_tokens":      MAX_TOKENS,
            "response_format": {"type": "json_object"},
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content":
                    f"Tafsiri yafuatayo kwa Kiswahili:\n\n"
                    f"MAAGIZO:\n{row['instruction']}\n\n"
                    f"JIBU:\n{row['output']}"
                },
            ],
        }
    }

with open(BATCH_JSONL, "w", encoding="utf-8") as f:
    for row in selected:
        f.write(json.dumps(make_request(row), ensure_ascii=False) + "\n")

size_mb = os.path.getsize(BATCH_JSONL) / 1e6
print(f"✅ Batch file: {BATCH_JSONL}")
print(f"   Rows: {len(selected):,} | Size: {size_mb:.1f} MB (limit: 100 MB)")

## 4. Submit Batch Job

⚠️ **Run this cell once.** The batch ID is saved to `batch_id.txt` — safe even if notebook restarts.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

# Upload file
print("Uploading batch file...")
with open(BATCH_JSONL, "rb") as f:
    batch_file = client.files.create(file=f, purpose="batch")
print(f"File uploaded: {batch_file.id}")

# Submit batch
batch_job = client.batches.create(
    input_file_id     = batch_file.id,
    endpoint          = "/v1/chat/completions",
    completion_window = "24h",
    metadata={"project": "lengai-lab", "dataset": "FineTome-20k-sw"},
)

BATCH_ID = batch_job.id

# Save to file — critical for notebook restarts
with open("batch_id.txt", "w") as f:
    f.write(BATCH_ID)

print(f"\n🚀 Batch submitted!")
print(f"   Batch ID : {BATCH_ID}")
print(f"   Status   : {batch_job.status}")
print(f"   Saved to : batch_id.txt")
print(f"\n💤 You can close this notebook and come back in 2–6 hours.")
print(f"   Track at : https://platform.openai.com/batches")

## 5. Poll for Completion

Run this after a few hours. It checks every 60s and stops when done.

In [ ]:
import time
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

# Load batch ID (handles notebook restarts)
if "BATCH_ID" not in globals():
    with open("batch_id.txt") as f:
        BATCH_ID = f.read().strip()
    print(f"Loaded Batch ID: {BATCH_ID}")

print(f"Polling {BATCH_ID} (every 60s)...\n")
elapsed = 0

while elapsed < 86400:  # 24h max
    batch  = client.batches.retrieve(BATCH_ID)
    counts = batch.request_counts
    print(f"[{elapsed//60:3d} min] {batch.status:12s} | "
          f"Total: {counts.total} | Done: {counts.completed} | Failed: {counts.failed}")

    if batch.status == "completed":
        OUTPUT_FILE_ID = batch.output_file_id
        with open("output_file_id.txt", "w") as f:
            f.write(OUTPUT_FILE_ID)
        print(f"\n🎉 Done! Output file: {OUTPUT_FILE_ID}")
        break
    elif batch.status in ("failed", "expired", "cancelled"):
        print(f"\n❌ Batch {batch.status}: {batch.errors}")
        break

    time.sleep(60)
    elapsed += 60

## 6. Download & Parse Results

In [ ]:
from openai import OpenAI
import json

client = OpenAI(api_key=OPENAI_API_KEY)

# Load output file ID if needed
if "OUTPUT_FILE_ID" not in globals():
    with open("output_file_id.txt") as f:
        OUTPUT_FILE_ID = f.read().strip()

# Download
print("Downloading results...")
result = client.files.content(OUTPUT_FILE_ID)
with open(RESULTS_JSONL, "wb") as f:
    f.write(result.content)
print(f"✅ Saved to {RESULTS_JSONL}")

# Build lookup: custom_id → original row
lookup = {f"row-{r['id']}": r for r in selected}

# Parse
translated, errors = [], []

with open(RESULTS_JSONL, encoding="utf-8") as f:
    for line in f:
        res      = json.loads(line)
        cid      = res["custom_id"]
        original = lookup.get(cid, {})

        if res.get("error"):
            errors.append({"id": cid, "error": res["error"]})
            continue
        try:
            content = res["response"]["body"]["choices"][0]["message"]["content"]
            parsed  = json.loads(content)
            instr_sw = parsed.get("instruction", "").strip()
            out_sw   = parsed.get("output", "").strip()

            if instr_sw and out_sw:
                translated.append({
                    "instruction":    instr_sw,
                    "output":         out_sw,
                    "instruction_en": original.get("instruction", ""),
                    "output_en":      original.get("output", ""),
                    "source":         "FineTome-100k",
                    "lang":           "sw",
                })
            else:
                errors.append({"id": cid, "error": "empty fields"})
        except Exception as e:
            errors.append({"id": cid, "error": str(e)})

print(f"\n✅ Parsed: {len(translated):,} translated | {len(errors):,} errors")
print(f"   Success rate: {100*len(translated)/max(len(translated)+len(errors),1):.1f}%")

## 7. Quality Validation

Filter out bad translations before publishing.

In [ ]:
import re

# Common Swahili function words as a presence signal
SW_MARKERS = [
    r'\bni\b', r'\bna\b', r'\bya\b', r'\bwa\b', r'\bkwa\b',
    r'\bkuwa\b', r'\bkama\b', r'\bkatika\b', r'\blakini\b',
    r'\bpia\b', r'\bsana\b', r'\bwakati\b', r'\bkwa hivyo\b',
    r'\bkwa mfano\b', r'\bkwa sababu\b', r'\bainayohusiana\b',
]

def has_swahili(text):
    t = text.lower()
    return sum(1 for p in SW_MARKERS if re.search(p, t)) >= 2

def length_ratio(src, tgt):
    s = max(len(src.split()), 1)
    return len(tgt.split()) / s

def validate(row):
    if not row["instruction"] or not row["output"]:
        return False, "empty"
    if not has_swahili(row["instruction"] + " " + row["output"]):
        return False, "no_swahili_markers"
    ratio = length_ratio(row["output_en"], row["output"])
    if not (0.5 <= ratio <= 2.5):
        return False, f"bad_length_ratio_{ratio:.2f}"
    if row["output"].strip() == row["output_en"].strip():
        return False, "untranslated"
    return True, "ok"

passed, failed = [], []
for row in translated:
    ok, reason = validate(row)
    (passed if ok else failed).append(row)

print(f"✅ Quality gate: {len(passed):,} passed | {len(failed):,} failed")
print(f"   Final yield: {100*len(passed)/max(len(translated),1):.1f}%")

# Show sample
if passed:
    s = passed[0]
    print(f"\n--- Sample ---")
    print(f"EN: {s['instruction_en'][:120]}")
    print(f"SW: {s['instruction'][:120]}")
    print(f"EN: {s['output_en'][:200]}")
    print(f"SW: {s['output'][:200]}")

## 8. Push to HuggingFace Hub

In [ ]:
from datasets import Dataset
from huggingface_hub import login

login(token=HF_TOKEN)

# Version 1 — instruction/output format (Alpaca-style)
ds_main = Dataset.from_list(passed)

# Version 2 — ShareGPT format (direct Unsloth SFT use)
ds_sharegpt = Dataset.from_list([
    {
        "conversations": [
            {"from": "human", "value": r["instruction"]},
            {"from": "gpt",   "value": r["output"]},
        ],
        "lang": "sw", "source": r["source"],
    }
    for r in passed
])

print(f"Pushing {len(passed):,} rows to HuggingFace...")

ds_main.push_to_hub(
    HF_REPO, token=HF_TOKEN,
    commit_message=f"Add {len(passed):,} Swahili-translated FineTome rows",
)
ds_sharegpt.push_to_hub(
    HF_REPO + "-sharegpt", token=HF_TOKEN,
    commit_message="ShareGPT format — ready for Unsloth SFT",
)

print(f"\n🎉 Published!")
print(f"   Main     → https://huggingface.co/datasets/{HF_REPO}")
print(f"   ShareGPT → https://huggingface.co/datasets/{HF_REPO}-sharegpt")
print(f"\n⚡ Next step: finetune Gemma4 E4B on this dataset")
print(f"   load_dataset('{HF_REPO}-sharegpt')")